# WOOD: corrected training and evaluation

This notebook uses the Python modules in the project folder as the single source of truth. The historical version is preserved in `notebooks/WOOD_original.ipynb`. Run from top to bottom. Training creates a new checkpoint; the original checkpoint is unchanged.


## Setup
In Colab, select a GPU runtime and copy the entire project folder to Drive. Edit `PROJECT` below if necessary. Locally, open this notebook with the project virtual environment as its kernel.


In [ ]:
import sys
import subprocess
from pathlib import Path
import os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/WOOD_MPE')
else:
    PROJECT = Path.cwd()
    if not (PROJECT / 'wood').is_dir() and (PROJECT.parent / 'wood').is_dir():
        PROJECT = PROJECT.parent
assert (PROJECT / 'wood').is_dir(), 'Set PROJECT to the extracted project folder'
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print(PROJECT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


## Verify device and regression checks
`auto` uses an available supported accelerator and otherwise CPU. An explicit CUDA selection fails if unavailable.


In [ ]:
from wood.config import get_device
print('Device:', get_device())
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## Train
The default split reserves training songs for validation. Thresholds are selected on complete validation songs after training. Use a new output folder for each experiment. For a short pipeline check set epochs to 1 and add `--max-batches 1 --thresholds 0.5`; that does not produce a useful trained model.


In [ ]:
RUN = 'runs/notebook'
subprocess.run([sys.executable, 'train.py', '--device', 'auto', '--epochs', '200',
                '--output', RUN], check=True)


## Training curves


In [ ]:
import json
import matplotlib.pyplot as plt
history = json.loads((Path(RUN) / 'history.json').read_text())
for key in ('train_loss', 'validation_loss'):
    plt.plot([r['epoch'] for r in history], [r[key] for r in history], label=key)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


## Evaluate all test songs
This uses saved validation thresholds and reports per-song and macro-averaged frame F1, note F1 and offset-aware note F1.


In [ ]:
subprocess.run([sys.executable, 'evaluate.py', '--checkpoint', f'{RUN}/best.pt',
                '--output', f'{RUN}/test_metrics.json'], check=True)


## Inspect one song


In [ ]:
from wood.engine import load_checkpoint
from wood.data import load_song, song_folders
from wood.decoding import predict_song
device = get_device()
model, cfg, checkpoint = load_checkpoint(f'{RUN}/best.pt', device)
song = load_song(song_folders('Data/test')[0], cfg)
th = checkpoint['thresholds']
pred = predict_song(model, song, cfg, device, th['onset'], th['offset'])
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, data, title in zip(axes, [song['frame'], pred['frame']], ['Reference', 'Prediction']):
    ax.imshow(data.T, aspect='auto', origin='lower')
    ax.set_title(title)
    ax.set_ylabel('Pitch index (MIDI minus 24)')
axes[-1].set_xlabel('Frame')
plt.tight_layout()
plt.show()
